# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [18]:
class matrix:


    def __init__(self, *args):

        if len(args) == 2:
            n = args[0]
            m = args[1]

            if not isinstance(n, int) or not isinstance(m, int):
                raise TypeError("Matrix dimensions must be integers.")

            if n < 0 or m < 0:
                raise ValueError("Matrix dimensions must be non-negative.")

            self.data = [[0.0 for j in range(m)] for i in range(n)]

        elif len(args) == 1:
            values = args[0]

            if not isinstance(values, list):
                raise TypeError("Matrix must be created from a list of lists.")

            if len(values) == 0:
                self.data = []
                return

            for row in values:
                if not isinstance(row, list):
                    raise ValueError("Every row must be a list.")

            number_of_columns = len(values[0])

            for row in values:
                if len(row) != number_of_columns:
                    raise ValueError(
                        "All rows must have the same number of columns."
                    )

            self.data = [list(row) for row in values]

        else:
            raise TypeError(
                "Use matrix(n, m) or matrix([[...], [...]])"
            )

   
    def shape(self):
        if len(self.data) == 0:
            return (0, 0)

        return (len(self.data), len(self.data[0]))

   
    def __getitem__(self, key):

        if isinstance(key, tuple):

            if len(key) != 2:
                raise IndexError("Use M[i,j]")

            row_key = key[0]
            col_key = key[1]

            if isinstance(row_key, int) and isinstance(col_key, int):
                return self.data[row_key][col_key]

            if isinstance(row_key, int):
                selected_rows = [self.data[row_key]]
            else:
                selected_rows = self.data[row_key]

            result = []

            for row in selected_rows:
                if isinstance(col_key, int):
                    result.append([row[col_key]])
                else:
                    result.append(row[col_key])

            return matrix(result)

        return self.data[key]

    
    def __setitem__(self, key, value):

        if isinstance(key, tuple):

            if len(key) != 2:
                raise IndexError("Use M[i,j]")

            row_key = key[0]
            col_key = key[1]

            if isinstance(row_key, int) and isinstance(col_key, int):
                self.data[row_key][col_key] = value
                return

            if isinstance(value, matrix):
                value = value.to_list()

            if not isinstance(value, list):
                raise TypeError("Assigned value must be a matrix or list.")

            if isinstance(row_key, int):
                row_indices = [row_key]
            else:
                row_indices = list(
                    range(*row_key.indices(self.shape()[0]))
                )

            if isinstance(col_key, int):
                col_indices = [col_key]
            else:
                col_indices = list(
                    range(*col_key.indices(self.shape()[1]))
                )

            if len(value) != len(row_indices):
                raise ValueError("Incorrect number of rows.")

            for row in value:
                if len(row) != len(col_indices):
                    raise ValueError("Incorrect number of columns.")

            for i in range(len(row_indices)):
                for j in range(len(col_indices)):
                    self.data[row_indices[i]][col_indices[j]] = value[i][j]

            return

        if isinstance(value, list):

            if len(value) != self.shape()[1]:
                raise ValueError("Incorrect number of columns.")

            self.data[key] = list(value)

        else:
            raise TypeError("A row must be assigned a list.")

   
    def copy_from(self, other):

        if not isinstance(other, matrix):
            raise TypeError("Can only copy from another matrix.")

        if self.shape() != other.shape():
            raise ValueError("Matrices must have the same dimensions.")

        self.data = other.to_list()

   
    def to_list(self):
        return [list(row) for row in self.data]

    
    def transpose(self):

        n, m = self.shape()

        return matrix([
            [self.data[i][j] for i in range(n)]
            for j in range(m)
        ])

    
    def row(self, n):

        if n < 0 or n >= self.shape()[0]:
            raise IndexError("Row index out of range.")

        return matrix([self.data[n]])

    
    def column(self, n):

        if n < 0 or n >= self.shape()[1]:
            raise IndexError("Column index out of range.")

        return matrix([
            [self.data[i][n]]
            for i in range(self.shape()[0])
        ])

  
    def block(self, n_0, n_1, m_0, m_1):

        n, m = self.shape()

        if n_0 < 0 or n_1 > n or n_0 > n_1:
            raise IndexError("Invalid row range.")

        if m_0 < 0 or m_1 > m or m_0 > m_1:
            raise IndexError("Invalid column range.")

        return matrix([
            row[m_0:m_1]
            for row in self.data[n_0:n_1]
        ])

    
    def scalarmul(self, c):

        return matrix([
            [c * value for value in row]
            for row in self.data
        ])

    
    def add(self, N):

        if not isinstance(N, matrix):
            raise TypeError("N must be a matrix.")

        if self.shape() != N.shape():
            raise ValueError(
                "Matrices must have the same dimensions."
            )

        n, m = self.shape()

        return matrix([
            [
                self.data[i][j] + N.data[i][j]
                for j in range(m)
            ]
            for i in range(n)
        ])

    
    def sub(self, N):

        if not isinstance(N, matrix):
            raise TypeError("N must be a matrix.")

        if self.shape() != N.shape():
            raise ValueError(
                "Matrices must have the same dimensions."
            )

        n, m = self.shape()

        return matrix([
            [
                self.data[i][j] - N.data[i][j]
                for j in range(m)
            ]
            for i in range(n)
        ])

    
    def mat_mult(self, N):

        if not isinstance(N, matrix):
            raise TypeError("N must be a matrix.")

        n, m = self.shape()
        p, q = N.shape()

        if m != p:
            raise ValueError(
                "The number of columns in the first matrix "
                "must equal the number of rows in the second."
            )

        result = []

        for i in range(n):

            new_row = []

            for j in range(q):

                total = 0

                for k in range(m):
                    total += self.data[i][k] * N.data[k][j]

                new_row.append(total)

            result.append(new_row)

        return matrix(result)

   
    def element_mult(self, N):

        if not isinstance(N, matrix):
            raise TypeError("N must be a matrix.")

        if self.shape() != N.shape():
            raise ValueError(
                "Matrices must have the same dimensions."
            )

        n, m = self.shape()

        return matrix([
            [
                self.data[i][j] * N.data[i][j]
                for j in range(m)
            ]
            for i in range(n)
        ])

    
    def equals(self, N):

        if not isinstance(N, matrix):
            return False

        return self.data == N.data

    
    def __mul__(self, other):

        if isinstance(other, (int, float)):
            return self.scalarmul(other)

        if isinstance(other, matrix):
            return self.mat_mult(other)

        return NotImplemented

    def __rmul__(self, other):

        if isinstance(other, (int, float)):
            return self.scalarmul(other)

        return NotImplemented

    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __eq__(self, other):
        return self.equals(other)

    def __repr__(self):
        return str(self.data)

In [21]:

M = matrix(2, 3)

print("M:")
print(M)

A = matrix([
    [1, 2, 3],
    [4, 5, 6]
])

print("\nA:")
print(A)

print("\nA[0][1]:")
print(A[0][1])

print("\nA[1,2]:")
print(A[1, 2])

print("\nShape of A:")
print(A.shape())

print("\nA as a list:")
print(A.to_list())

M:
[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]

A:
[[1, 2, 3], [4, 5, 6]]

A[0][1]:
2

A[1,2]:
6

Shape of A:
(2, 3)

A as a list:
[[1, 2, 3], [4, 5, 6]]


In [22]:
B = matrix([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])

print("Transpose:")
print(B.transpose())

print("\nRow 1:")
print(B.row(1))

print("\nColumn 2:")
print(B.column(2))

print("\nBlock:")
print(B.block(0, 2, 1, 3))

print("\nSlice:")
print(B[0:2, 1:3])

def constant(n, m, c):

    return matrix([
        [float(c) for j in range(m)]
        for i in range(n)
    ])


def zeros(n, m):
    return constant(n, m, 0)


def ones(n, m):
    return constant(n, m, 1)


def eye(n):

    return matrix([
        [
            1.0 if i == j else 0.0
            for j in range(n)
        ]
        for i in range(n)
    ])


print("\nConstant:")
print(constant(2, 3, 5))

print("\nZeros:")
print(zeros(2, 2))

print("\nOnes:")
print(ones(2, 2))

print("\nIdentity:")
print(eye(3))

C = matrix([
    [1, 2],
    [3, 4]
])

D = matrix([
    [5, 6],
    [7, 8]
])

print("\nScalar multiplication:")
print(C.scalarmul(2))

print("\nAddition:")
print(C.add(D))

print("\nSubtraction:")
print(C.sub(D))

print("\nMatrix multiplication:")
print(C.mat_mult(D))

print("\nElement-wise multiplication:")
print(C.element_mult(D))

print("\nC equals C:")
print(C.equals(C))

print("\nC equals D:")
print(C.equals(D))

print("\n2 * C:")
print(2 * C)

print("\nC * 2:")
print(C * 2)

print("\nC + D:")
print(C + D)

print("\nC - D:")
print(C - D)

print("\nC * D:")
print(C * D)

print("\nC == C:")
print(C == C)

print("\nC == D:")
print(C == D)

A = matrix([
    [1, 2],
    [3, 4]
])

B = matrix([
    [2, 0],
    [1, 2]
])

C = matrix([
    [0, 1],
    [2, 3]
])

I = eye(2)

left_side = (A * B) * C
right_side = A * (B * C)

print("\n(AB)C:")
print(left_side)

print("\nA(BC):")
print(right_side)

print("\n(AB)C == A(BC):")
print(left_side == right_side)

left_side = A * (B + C)
right_side = (A * B) + (A * C)

print("\nA(B+C):")
print(left_side)

print("\nAB + AC:")
print(right_side)

print("\nA(B+C) == AB + AC:")
print(left_side == right_side)

AB = A * B
BA = B * A

print("\nAB:")
print(AB)

print("\nBA:")
print(BA)

print("\nAB == BA:")
print(AB == BA)

print("\nAB != BA:")
print(AB != BA)

AI = A * I

print("\nAI:")
print(AI)

print("\nA:")
print(A)

print("\nAI == A:")
print(AI == A)

Transpose:
[[1, 4, 7], [2, 5, 8], [3, 6, 9]]

Row 1:
[[4, 5, 6]]

Column 2:
[[3], [6], [9]]

Block:
[[2, 3], [5, 6]]

Slice:
[[2, 3], [5, 6]]

Constant:
[[5.0, 5.0, 5.0], [5.0, 5.0, 5.0]]

Zeros:
[[0.0, 0.0], [0.0, 0.0]]

Ones:
[[1.0, 1.0], [1.0, 1.0]]

Identity:
[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]

Scalar multiplication:
[[2, 4], [6, 8]]

Addition:
[[6, 8], [10, 12]]

Subtraction:
[[-4, -4], [-4, -4]]

Matrix multiplication:
[[19, 22], [43, 50]]

Element-wise multiplication:
[[5, 12], [21, 32]]

C equals C:
True

C equals D:
False

2 * C:
[[2, 4], [6, 8]]

C * 2:
[[2, 4], [6, 8]]

C + D:
[[6, 8], [10, 12]]

C - D:
[[-4, -4], [-4, -4]]

C * D:
[[19, 22], [43, 50]]

C == C:
True

C == D:
False

(AB)C:
[[8, 16], [16, 34]]

A(BC):
[[8, 16], [16, 34]]

(AB)C == A(BC):
True

A(B+C):
[[8, 11], [18, 23]]

AB + AC:
[[8, 11], [18, 23]]

A(B+C) == AB + AC:
True

AB:
[[4, 4], [10, 8]]

BA:
[[2, 4], [7, 10]]

AB == BA:
False

AB != BA:
True

AI:
[[1.0, 2.0], [3.0, 4.0]]

A:
[[1, 2]